# 🔌 hermes-acp-sdk — start here

**`hermes-acp-sdk`** lets *any* Python app drive the **Hermes Agent** over the
Agent Client Protocol (ACP) — a CLI, a web backend, a bot. No Jupyter needed
(this notebook is just a convenient place to show it).

> ✅ **Everything in THIS notebook runs right now — no API key, no Hermes, zero tokens.**
> The parts that need a live agent are in **`01_showcase.ipynb`**, which is saved
> *with its real output* so you can read it without spending anything.

**What you'll do here**
1. See the typed event model the SDK gives you
2. Watch the **security model** refuse a hostile agent — live
3. Give an agent **your own Python function** as a tool, and call it — live

```bash
pip install hermes-acp-sdk          # this SDK
pip install "hermes-agent[acp]"     # the agent it drives (needs Python <3.14)
```

## 0 · Import — no API key needed

In [1]:
import hermes_acp_sdk
from hermes_acp_sdk import (
    HermesClient, HermesSession,                       # driving the agent
    AgentText, AgentThought, ToolCall, PlanUpdated,    # the typed events
    Usage, PermissionDenied, Finished,
    DenyAll, AllowAll, AllowTools, CallbackPolicy,     # the security policies
    FsPolicy, TerminalPolicy,
    ToolServer,                                        # your functions as agent tools
    HermesNotFound, EventHandler,
)
print("hermes-acp-sdk", hermes_acp_sdk.__version__, "—", len(hermes_acp_sdk.__all__), "public names")

hermes-acp-sdk 0.1.1 — 28 public names


## 1 · 🔒 Safe by default — watch it refuse a hostile agent

An ACP agent may ask the **client** to read your files or run shell commands. That is the
sharp edge of this protocol. This SDK says **no** unless you opt in — on purpose.

*(Runs live, zero tokens: we call the very callbacks the agent would.)*

In [2]:
handler = EventHandler(DenyAll(), FsPolicy(), TerminalPolicy())     # the DEFAULTS

try:
    await handler.read_text_file(path="/etc/passwd", session_id="s1")
except PermissionError as e:
    print("❌ agent tried to read /etc/passwd →", e)

try:
    await handler.create_terminal(command="rm", session_id="s1", args=["-rf", "/"])
except PermissionError as e:
    print("❌ agent tried to run `rm -rf /`   →", e)

# Flipping the switch is NOT a blank cheque — you must name the commands.
loose = EventHandler(DenyAll(), FsPolicy(), TerminalPolicy(enabled=True))
try:
    await loose.create_terminal(command="curl", session_id="s1")
except PermissionError as e:
    print("❌ terminals ON, but `curl` not allow-listed →", e)

❌ agent tried to read /etc/passwd → read access to '/etc/passwd' is not allowed
❌ agent tried to run `rm -rf /`   → terminal access is not allowed (TerminalPolicy.enabled is False)
❌ terminals ON, but `curl` not allow-listed → command 'curl' is not in TerminalPolicy.allowed_commands ([]) — refusing to run it


And even a command you *did* allow does **not** inherit your environment — otherwise your
`DEEPSEEK_API_KEY` would be handed straight to a process the **agent** chose:

In [3]:
import os
os.environ["MY_API_KEY"] = "super-secret-value"      # pretend this is your real key

safe = EventHandler(
    DenyAll(), FsPolicy(),
    TerminalPolicy(enabled=True, allowed_commands=frozenset({"/bin/sh"})),
)
r = await safe.create_terminal(
    command="/bin/sh", session_id="s1",
    args=["-c", "echo the agent sees MY_API_KEY=[$MY_API_KEY]"],
)
await safe.wait_for_terminal_exit(session_id="s1", terminal_id=r.terminal_id)
out = await safe.terminal_output(session_id="s1", terminal_id=r.terminal_id)
print("✅ the allow-listed command ran:", out.output.strip())
print("   → your secret did NOT leak into it.")

✅ the allow-listed command ran: the agent sees MY_API_KEY=[]
   → your secret did NOT leak into it.


A permission request is refused **and reported to you** as an event, so nothing happens silently:

In [4]:
from types import SimpleNamespace

tool_call = SimpleNamespace(tool_call_id="tc1", title="Delete The Database")
option    = SimpleNamespace(option_id="allow-once", name="Allow", kind="allow_once")

resp = await handler.request_permission(options=[option], session_id="s1", tool_call=tool_call)
print("agent asked to run:", tool_call.title)
print("  outcome  :", type(resp.outcome).__name__)
print("  you get  :", handler.queue_for("s1").get_nowait())

agent asked to run: Delete The Database
  outcome  : DeniedOutcome
  you get  : PermissionDenied(tool_title='Delete The Database')


## 2 · 🛠 Give the agent **your own Python function** as a tool

An agent only knows what its tools let it know — and it knows **nothing** about *your*
database, *your* users, *your* state.

`ToolServer` runs an MCP server **inside your own process**, so the tools are ordinary
Python functions with full access to your application. *(This is the key to "the coach
remembers the student".)*

Below we serve a function and call it with a **real MCP client** — still **zero tokens**.

In [5]:
from mcp.client.session import ClientSession
from mcp.client.streamable_http import streamablehttp_client

# Pretend this is your app's live state
ERROR_HISTORY = {"bex": "IndexError (9x), NameError (3x)", "ann": "KeyError (5x)"}

def student_weakness(student_id: str) -> str:
    "Look up which Python errors a given student most often gets wrong."
    return ERROR_HISTORY.get(student_id, "no history for that student")

async with ToolServer([student_weakness], name="coach-tools") as tools:
    cfg = tools.mcp_config
    print("MCP server  :", cfg.url)
    print("auth        :", cfg.headers[0].name, "= Bearer *** (only the agent gets it)")

    headers = {h.name: h.value for h in cfg.headers}
    async with streamablehttp_client(cfg.url, headers=headers) as (r, w, _):
        async with ClientSession(r, w) as mcp:
            await mcp.initialize()
            listed = await mcp.list_tools()
            print("tool exposed:", listed.tools[0].name)
            print("description :", listed.tools[0].description)
            result = await mcp.call_tool("student_weakness", {"student_id": "bex"})
            print("CALL RESULT :", result.content[0].text)

[07/13/26 14:24:06] INFO     StreamableHTTP session manager started                  streamable_http_manager.py:131

MCP server  : http://127.0.0.1:52314/mcp
auth        : Authorization = Bearer *** (only the agent gets it)


                    INFO     Terminating session: None                                       streamable_http.py:788

                    INFO     HTTP Request: POST http://127.0.0.1:52314/mcp "HTTP/1.1 200 OK"        _client.py:1740

                    INFO     Negotiated protocol version: 2025-11-25                         streamable_http.py:193

...

                    INFO     Processing request of type CallToolRequest                               server.py:733

                    INFO     Terminating session: None                                       streamable_http.py:788

                    INFO     HTTP Request: POST http://127.0.0.1:52314/mcp "HTTP/1.1 200 OK"        _client.py:1740

CALL RESULT : IndexError (9x), NameError (3x)


                    INFO     StreamableHTTP session manager shutting down            streamable_http_manager.py:135

The function's **name, type hints and docstring** became the tool's name, schema and
description automatically — that docstring is what the model reads when deciding to call it.

The server is **loopback-only and bearer-token protected**, so no other process on the
machine can invoke your functions:

In [6]:
import httpx

async with ToolServer([student_weakness]) as tools:
    async with httpx.AsyncClient() as http:
        resp = await http.post(tools.mcp_config.url, json={}, timeout=10)
    print("a local process with no token gets:", resp.status_code, "Unauthorized ✅")

                    INFO     StreamableHTTP session manager started                  streamable_http_manager.py:131

                    INFO     HTTP Request: POST http://127.0.0.1:52317/mcp "HTTP/1.1 401            _client.py:1740
                             Unauthorized"                                                                         

a local process with no token gets: 401 Unauthorized ✅


[07/13/26 14:24:07] INFO     StreamableHTTP session manager shutting down            streamable_http_manager.py:135

## 3 · 📨 The events you get back

`session.prompt(text)` is an **async iterator** of typed, frozen dataclasses — streaming
becomes an ordinary `for` loop:

| Event | Meaning |
|---|---|
| `AgentText` | a chunk of the agent's visible reply |
| `AgentThought` | a chunk of its **reasoning** — you can show it or hide it |
| `ToolCall` | a tool call started / changed status |
| `PlanUpdated` | the agent published or revised its plan |
| `Usage` | token accounting |
| `PermissionDenied` | your policy refused something |
| `Finished` | always last — the turn is over |

In [7]:
# The events are plain frozen dataclasses — easy to match on, easy to test.
print(AgentText(text="hello"))
print(AgentThought(text="hmm, let me think"))
print(Finished(stop_reason="end_turn"))

AgentText(text='hello')
AgentThought(text='hmm, let me think')
Finished(stop_reason='end_turn')


## 4 · 🧭 Talking to a real Hermes

That needs the agent itself plus a provider key:

```bash
pip install "hermes-agent[acp]"        # Python >=3.11,<3.14 !
export DEEPSEEK_API_KEY=sk-...
```

```python
async with HermesClient(model=MODEL) as hermes:          # spawns `hermes acp`
    async with hermes.session() as s:         # + selects a model (the classic trap)
        async for ev in s.prompt("What is a Python traceback?"):
            if isinstance(ev, AgentText):
                print(ev.text, end="")
```

👉 **See `01_showcase.ipynb`** — it is saved **with the real output of a live Hermes**,
so you can read the whole conversation, the agent's thoughts and the token usage
**without a key and without spending anything.**

If a `hermes` binary is present here, the cell below will actually run it; otherwise it
tells you so and moves on.

In [8]:
import shutil, sys
from pathlib import Path

have_hermes = bool(shutil.which("hermes") or (Path(sys.executable).parent / "hermes").exists())
have_key = bool(os.environ.get("DEEPSEEK_API_KEY"))

if not (have_hermes and have_key):
    print("ℹ️  No live Hermes here (hermes binary:", have_hermes, "| provider key:", have_key, ")")
    print("    That's expected on a shared lab — see 01_showcase.ipynb for the saved real run.")
else:
    async with HermesClient(model=MODEL) as hermes:
        print("connected to:", hermes.agent_name, hermes.agent_version)
        async with hermes.session() as s:
            async for ev in s.prompt("In one short sentence: what is a Python traceback?"):
                if isinstance(ev, AgentText):
                    print(ev.text, end="")
                elif isinstance(ev, Finished):
                    print(f"\n[{ev.stop_reason}]")

ℹ️  No live Hermes here (hermes binary: True | provider key: False )
    That's expected on a shared lab — see 01_showcase.ipynb for the saved real run.


---
### What you just saw
- The SDK gives you a **typed event stream** instead of a raw two-way protocol
- It is **locked down by default** — files, terminals and permissions all refused
- Your **own Python functions** become agent tools, running in your process with your state

**Next:** `01_showcase.ipynb` — the full tour against a live Hermes (output saved).
Source: <https://github.com/VoixKz/hermes-acp-sdk> · `pip install hermes-acp-sdk`